# Hands-on: Bring observations into an Anemoi model

So far our models learnt from **reanalysis** (ERA5): complete, gridded fields. In this session we take the deterministic model of session 5 and teach it to use **observations**: real measurements, at irregular places and times, with large gaps.

We use the **multi-dataset** support of anemoi: besides the ERA5 dataset `data`, the model gets a second dataset, `obs`, with its own nodes in the graph and its own encoder and/or decoder. Depending on where the observations enter the model, we get three different setups:

| | Setup | Observations are | What it is good for |
|---|---|---|---|
| **1** | **Decode to observations** (main part of this session) | a *target*: a new decoder predicts them | predict the observed quantities wherever we want, including variables the model never had |
| **2** | **Encode observations** | an *input*: a new encoder reads them | correct the forecast with the latest observations: a first step towards data assimilation |
| **3** | **Observations only** (like [AIFS-DOP](https://arxiv.org/abs/2412.15687)) | input *and* target, no ERA5 at all | a forecast learnt from observations only, without reanalysis |

```
   1: decode to obs                         2: encode obs                         3: obs only
data ─enc─► hidden ─proc─► dec ─► data    data ─enc─┐                        obs_in ─enc─► hidden ─proc─► dec ─► obs_out
                            └─dec─► obs   obs  ─enc─┴(+)─► hidden ─proc─► dec ─► data
```

**anemoi-core `main` vs. the next release.** This notebook runs on the current `main` branch of anemoi-core, which only supports **gridded** datasets: every dataset is a field on a fixed set of points. We therefore use observations that are *binned* onto a grid: a grid point holds the observation made there (within the 6-hour window), or NaN. The next release ("richer batches", section 8) adds **tabular** observations: each sample holds the observations there are, with their own positions and times. Everything we set up here (datasets, graph, encoders and decoders, losses, fine-tuning) carries over.

**Resources**
- [anemoi-training: multiple datasets](https://anemoi.readthedocs.io/projects/training/en/latest/)
- [anemoi-models: multi-dataset model configs](https://anemoi.readthedocs.io/projects/models/en/latest/)
- [Alexe et al. 2024, GraphDOP: Towards skilful data-driven medium-range weather forecasts learnt and initialised directly from observations](https://arxiv.org/abs/2412.15687)

## 0. Setup

This session needs anemoi-core `main` (anemoi-training ≥ 0.10, anemoi-models ≥ 0.13): many config keys changed since the version of session 5, e.g. `task:`, `training.optimization` and `training.checkpoint`. Use the kernel of this session, not the one of session 5.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from importlib.metadata import version
from pathlib import Path

import cartopy.crs as ccrs
import matplotlib.pyplot as plt
import mlflow
import numpy as np
import pandas as pd
import torch
from omegaconf import OmegaConf

from helpers import (config_diff, latest_run, load_config, open_datasets, plot_metrics, predict, run_training,
                     scatter_map, target)

for package in ["anemoi-training", "anemoi-models", "anemoi-graphs", "anemoi-datasets"]:
    print(f"{package:16} {version(package)}")
pd.set_option("display.float_format", "{:.3g}".format)  # 3 significant digits in tables
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none, training will be very slow")

In [ ]:
# The ERA5 toy dataset of session 5 (O48, 6-hourly, 2010-2022)
DATASET = "/datasets/aifs-ea-an-oper-0001-mars-o48-2010-2022-6h-v1-toy.zarr"
# The gridded observation dataset. For testing we use radiosondes binned onto the O96 grid (from the AIFS-DOP training data).
OBS_DATASET = "/datasets/aifsdop-ea-ofb-oper-0001-mars-o96-2010-2024-6h-v1-observations-radiosondes.zarr"
# The model we fine-tune: a checkpoint of configs/pretrain_o48.yaml (the deterministic model of session 5, trained with anemoi-core main).
# It must be a *training* checkpoint (last.ckpt), not an inference checkpoint. If you do not have one, train it in section 5.0.
PRETRAINED = "/datasets/checkpoints/pretrained-o48.ckpt"
# Where the trainings write their output
OUTPUT_ROOT = Path.home() / "anemoi-output"

OVERRIDES = [f"system.input.dataset={DATASET}", f"system.input.obs_dataset={OBS_DATASET}", f"system.output.root={OUTPUT_ROOT}/"]
FINETUNE = [f"training.checkpoint.source.path={PRETRAINED}"]  # for the configs that start from the pretrained model

## 1. The observation dataset

The observation dataset is an ordinary anemoi dataset (a Zarr store), so we open it with `open_dataset` like the ERA5 one.

In [ ]:
from anemoi.datasets import open_dataset

era5 = open_dataset(DATASET)
obs_all = open_dataset(OBS_DATASET)
print(f"ERA5: {era5.shape} on {era5.resolution}, {era5.dates[0]} to {era5.dates[-1]}, every {era5.frequency}")
print(f"obs:  {obs_all.shape} on {obs_all.resolution}, {obs_all.dates[0]} to {obs_all.dates[-1]}, every {obs_all.frequency}")
print("obs variables:", obs_all.variables)

The radiosonde dataset holds temperature `t`, humidity `q`, wind `u`, `v` and geopotential `z` on 13 pressure levels, plus variables known everywhere: coordinates, time, insolation, land-sea mask, orography (`z`) and some properties of the soundings (`sonde_*`). We use a few of the observed variables and the forcings, selected with `select` when opening the dataset (the same option goes into the training config):

In [ ]:
OBSERVED = ["t_850", "t_500", "z_500", "u_500", "v_500", "q_700"]
FORCINGS = ["cos_latitude", "cos_longitude", "sin_latitude", "sin_longitude", "cos_julian_day", "cos_local_time",
            "sin_julian_day", "sin_local_time", "insolation", "lsm", "z"]
obs = open_dataset(OBS_DATASET, select=FORCINGS + OBSERVED)
print(obs.variables)

date = np.datetime64("2021-03-01T12:00")
values = obs[int(np.where(obs.dates == date)[0][0])]
for v in OBSERVED + ["lsm"]:
    print(f"{v:6} {np.isnan(values[obs.name_to_index[v], 0]).mean():6.1%} NaN")

At a given time, almost every grid point is **NaN**: there was no sounding there. The forcings have no gaps. Let's look at where the observations are.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 4.5), subplot_kw={"projection": ccrs.PlateCarree()})
for ax, d in zip(axes, [date, date + np.timedelta64(6, "h")]):
    i = int(np.where(obs.dates == d)[0][0])
    t850 = obs[i][obs.name_to_index["t_850"], 0]
    im = scatter_map(ax, obs.latitudes, obs.longitudes, t850, title=f"t_850 observations at {d} ({(~np.isnan(t850)).sum()} points)",
                     s=8, cmap="RdYlBu_r")
plt.colorbar(im, ax=axes, shrink=0.8, label="K")

# number of observed grid points over 4 days
idx = np.where((obs.dates >= date) & (obs.dates < date + np.timedelta64(4, "D")))[0]
count = [(~np.isnan(obs[i][obs.name_to_index["t_850"], 0])).sum() for i in idx]
plt.figure(figsize=(8, 2.5))
plt.bar([str(d)[5:13] for d in obs.dates[idx]], count)
plt.xticks(rotation=90)
plt.ylabel("observed points (t_850)")

**Questions**
- Why are there many more soundings at 00 and 12 UTC than at 06 and 18 UTC?
- Where are the gaps? What does this mean for what a model can learn from these observations?

### Normalisation
Like any other variable, the observations are normalised with the statistics of the dataset. These are computed over the observations only (the NaNs are ignored), so they can differ from the ERA5 statistics of the same variable:

In [ ]:
stats = pd.DataFrame({
    "obs mean": [obs.statistics["mean"][obs.name_to_index[v]] for v in ["t_850", "z_500"]],
    "ERA5 mean": [era5.statistics["mean"][era5.name_to_index[v]] for v in ["t_850", "z_500"]],
    "obs stdev": [obs.statistics["stdev"][obs.name_to_index[v]] for v in ["t_850", "z_500"]],
    "ERA5 stdev": [era5.statistics["stdev"][era5.name_to_index[v]] for v in ["t_850", "z_500"]],
}, index=["t_850", "z_500"])
stats

**Question:** the sondes are mostly launched over land in the northern hemisphere. How does that show in the statistics?

### Matching the dates of the datasets

In every training sample, the model sees ERA5 and the observations **at the same times**. On anemoi-core `main`, the samples of the datasets are matched by their **position** in the selected period, not by their date. So every dataset must have the same frequency and the same first date. Look at the first dates:

In [ ]:
print("ERA5 starts at", era5.dates[0])
print("obs  starts at", obs.dates[0])

# what we would get without care: sample k of each dataset
for k in range(3):
    print(f"sample {k}: ERA5 {era5.dates[k]}  obs {obs.dates[k]}")

They are 6 hours apart, which nothing would warn us about! That is why the dataloader config of this session starts **all** datasets on `2010-01-02` (see `configs/dataloader/toy_o48.yaml`), and the obs config (`configs/obs/radiosondes.yaml`) takes `start` and `end` from the ERA5 dataset. Check it:

In [ ]:
for k in range(3):
    print(f"sample {k}: ERA5 {open_dataset(DATASET, start='2010-01-02').dates[k]}  obs {open_dataset(OBS_DATASET, start='2010-01-02').dates[k]}")

## 2. A model with two datasets: the config

All configs of this session are in `configs/`. Files that are not there (e.g. `model/graphtransformer.yaml`) come from the installed anemoi-training. The **pretrained model** is the deterministic model of session 5:

| Config | What it is |
|---|---|
| `pretrain_o48.yaml` | the pretrained model: ERA5 only (dataset `data`) |
| `finetune_obs_decoder.yaml` | **setup 1**: pretrained model + decoder to `obs` |
| `encode_obs.yaml` | **setup 2**: pretrained model + encoder of `obs` |
| `dop_obs_only.yaml` | **setup 3**: observations only, from scratch |
| `obs/radiosondes.yaml` | everything about the observation dataset, used by setups 1 and 2 |

With several datasets, most sections of the config have one entry per dataset, keyed by the **dataset name** (`data`, `obs`). The name is also the name of the dataset's nodes in the graph.

| Section | Per dataset |
|---|---|
| `data.datasets.<name>` | which variables are forcings or diagnostics, normalisation, imputation |
| `dataloader.{training,validation,test}.datasets.<name>` | which zarr, which variables, which dates |
| `graph.nodes.<name>` | the nodes of the dataset (its grid points) |
| `graph.edges` | edges `<name> → hidden` (encoder) and `hidden → <name>` (decoder) |
| `model.encoders.<group>.source_datasets` | which datasets an encoder reads |
| `model.decoders.<group>.target_datasets` | which datasets a decoder predicts |
| `training.training_loss.datasets.<name>` | loss of the dataset (no entry: no loss) |
| `training.scalers.datasets.<name>`, `training.metrics.datasets.<name>`, ... | loss weights and validation metrics |

Encoders and decoders are organised in **groups** with a name of your choice (`"0"` in the default model config). The name appears in the names of the model weights, so the checkpoint of the pretrained model has `encoder.0` and `decoder.0`, and our new decoder will be `decoder.obs`.

Here is everything about the observation dataset:

In [ ]:
print(Path("configs/obs/radiosondes.yaml").read_text())

The **role of the variables** decides what the observations are to the model:
- `diagnostic`: output only. The model predicts them but does not get them as input.
- `forcing`: input only.
- all others are `prognostic`: input and output.

In setup 1, the observed variables are **diagnostic**: a dataset with only diagnostic variables and forcings is an *output-only* dataset. It needs a decoder, but no encoder.

The **imputer** fills the gaps. On `main` a dataset must not have NaNs after its processors, so the gaps are filled with the mean of each variable. The filled-in values must not count in the loss! anemoi's `InputImputer` gives the loss a mask (the `nan_mask_weights` scaler), but it takes the gaps of the *first* time step of the sample (an input time), which is right for gaps that never change (the land points of a sea-surface temperature field) but not for observations. Our `ObsImputer` (in `obs_imputer.py`) takes the gaps of the *target* time instead. With tabular observations (section 8) there are no gaps to fill.

In [ ]:
print(Path("obs_imputer.py").read_text())

Now let's compare the full configs of the pretrained model and of setup 1. `config_diff` lists every setting that differs:

In [ ]:
pretrain = load_config("pretrain_o48", OVERRIDES)
finetune = load_config("finetune_obs_decoder", OVERRIDES + FINETUNE)
pd.set_option("display.max_colwidth", 120)
pd.DataFrame(config_diff(pretrain, finetune), columns=["setting", "pretrain_o48", "finetune_obs_decoder"])

Besides the obs entries, note `config_validation: False`: the config schema of anemoi-training only knows its own processors, and we use our own imputer.

## 3. The graph

The graph gets a third set of nodes: the 40,320 points of the O96 observation grid. The decoder of the observations needs edges from the hidden mesh to them: each observation node receives from its 3 nearest hidden nodes (`KNNEdges`).

In [ ]:
print(OmegaConf.to_yaml(finetune.graph.edges[-1]))

In [ ]:
from anemoi.graphs.create import GraphCreator

graph = GraphCreator(config=finetune.graph).create()
print(graph)

In [ ]:
def lonlat(nodes):
    lat, lon = np.rad2deg(graph[nodes].x.cpu().numpy()).T
    return (lon + 180) % 360 - 180, lat


fig, ax = plt.subplots(figsize=(14, 6), subplot_kw={"projection": ccrs.PlateCarree()})
ax.scatter(*lonlat("obs"), s=2, c="tab:green", label=f"obs nodes ({graph['obs'].num_nodes})")
ax.scatter(*lonlat("data"), s=6, c="grey", label=f"data nodes ({graph['data'].num_nodes})")
ax.scatter(*lonlat("hidden"), s=14, c="tab:red", label=f"hidden nodes ({graph['hidden'].num_nodes})")
# the decoder edges of a few observation nodes
src, dst = graph[("hidden", "to", "obs")].edge_index.cpu().numpy()
hlon, hlat = lonlat("hidden")
olon, olat = lonlat("obs")
for k in np.where((olat[dst] > 45) & (olat[dst] < 47) & (olon[dst] > 0) & (olon[dst] < 10))[0]:
    ax.plot([hlon[src[k]], olon[dst[k]]], [hlat[src[k]], olat[dst[k]]], c="k", lw=0.8)
ax.coastlines()
ax.set_extent([-10, 25, 35, 60])
ax.legend(loc="lower left")

**Question:** the decoder predicts the observations at all 40,320 obs nodes, but the loss only sees the few hundred observed ones of each sample. Is it a problem that the graph has nodes where there is never an observation (e.g. the oceans)?

## 4. The model: a new decoder

The model section of setup 1 adds a decoder group `obs`:

In [ ]:
print(OmegaConf.to_yaml({"decoders": {"obs": {k: v for k, v in finetune.model.decoders.obs.items() if k != "mapper"}}}))
print("trainable parameters per edge of the obs decoder:", finetune.model.decoders.obs.mapper.trainable_size)
print("trainable parameters per obs node:", finetune.model.node_trainable_parameters.obs)

Two choices matter here:

- **`target_node_features`**: what the decoder knows about each point it predicts, besides the hidden state. The ERA5 decoder uses `encoded_data`, the state of the point after the encoder: there is none for `obs`, which has no encoder. Instead, we give the decoder the **coordinates** and the **forcings** of the point (time of day, insolation, land-sea mask, orography).
- **No trainable parameters per node or edge** (`trainable_size: 0`, `node_trainable_parameters.obs: 0`). With them, the decoder would learn something specific to every observation location. Without them, it is a function of the hidden state and of the properties of the point only: it can decode **anywhere**, also where it never saw an observation, and later at any new location.

## 5. Fine-tune

### 5.0 The pretrained model (only if you do not have one)

The pretrained model is the deterministic model of session 5, trained with anemoi-core `main` for 2000 steps (about 8 minutes). Skip this if `PRETRAINED` exists.

In [ ]:
if not Path(PRETRAINED).exists():
    run_training("pretrain_o48", OVERRIDES)
    client = mlflow.tracking.MlflowClient(tracking_uri=str(OUTPUT_ROOT / "logs" / "mlflow"))
    PRETRAINED = str(OUTPUT_ROOT / "checkpoint" / latest_run(client, "pretrain-o48").info.run_id / "last.ckpt")
    FINETUNE = [f"training.checkpoint.source.path={PRETRAINED}"]
print(PRETRAINED)

### 5.1 Starting from a checkpoint: `training.checkpoint`

On `main`, how a training starts from a checkpoint is set in `training.checkpoint`, in three stages:

| Stage | Config group | Here |
|---|---|---|
| `source`: where the checkpoint is | `training/checkpoint/source`: `local`, `run` (an earlier run id), `s3`, `http` | `local`: the file `PRETRAINED` |
| `loading`: what to take from it | `training/checkpoint/loading`: `warm_start` (resume the same run), `weights_only`, `cold_start`, `transfer_learning` | `transfer_learning`: every weight that exists in both models with the same shape. Our new decoder does not exist in the checkpoint, so it starts from random weights. |
| `modifiers`: what to do with the model then | `training/checkpoint/modifiers`: `freezing` | freeze everything that was pretrained |

In [ ]:
print(OmegaConf.to_yaml(finetune.training.checkpoint))

With everything pretrained frozen, the forecast of the model stays exactly as it was: the new decoder learns to read the observed quantities from the hidden state of the pretrained model.

**Question:** we also freeze `model.model.node_attributes` and the `*_graph_provider` modules. What are they? (Hint: `model.node_trainable_parameters` and `model.edge_trainable_parameters` in the config.)

### 5.2 Train

1000 steps take about **5 minutes** on the course GPU (on an A100). In the output, look for the lines of the checkpoint pipeline: how many weights were loaded, which modules were frozen, and that `obs` was initialised randomly. In the model summary, compare the trainable and the non-trainable parameters.

In [ ]:
run_training("finetune_obs_decoder", OVERRIDES + FINETUNE)

## 6. Results

### 6.1 Loss curves

In [ ]:
client = mlflow.tracking.MlflowClient(tracking_uri=str(OUTPUT_ROOT / "logs" / "mlflow"))
run = latest_run(client, "finetune-obs-decoder")
print(f"run {run.info.run_name} ({run.info.run_id}), status {run.info.status}")
print(sorted(m for m in run.data.metrics if m.startswith("val_")))

In [ ]:
plot_metrics(client, [run], ["train_multi_dataset_loss_epoch", "val_obs_lossvariablemapper_loss", "val_data_lossvariablemapper_loss"])

**What to look for**
- The **obs** validation loss falls: the new decoder learns.
- The **data** (ERA5) validation loss is flat: the frozen model does not change.

The validation metrics of `obs` in MLflow (`val_obs_mse_metric/obs/...`) are **not** the mean squared error over the observations: the area weights (`node_weights`) sum to 1 over all 40,320 obs nodes, of which only ~1% are observed, so the metric grows with the number of observations. Fine for following the training, but for the skill we compute the errors ourselves.

### 6.2 The predictions against the observations

We run the fine-tuned model ourselves, on a date of the validation year (2021), like a forecast: load the **inference checkpoint**, give it the inputs (ERA5 at *t*-6h and *t*), and compare the predicted observations at *t*+6h with the radiosondes.

In [ ]:
checkpoint = OUTPUT_ROOT / "checkpoint" / run.info.run_id / "inference-last.ckpt"
model = torch.load(checkpoint, map_location="cuda", weights_only=False).cuda().eval()
print("datasets:", list(model.data_indices))

# the datasets, opened exactly as in the training (dataloader.validation.datasets.<name>.dataset_config)
datasets = open_datasets({name: cfg.dataset_config for name, cfg in finetune.dataloader.validation.datasets.items()})

forecast_date = "2021-03-01T06:00"  # the forecast is valid 6 hours later, at 12 UTC: many soundings
prediction = predict(model, datasets, forecast_date)
truth = target(datasets, "obs", forecast_date)
print({name: list(p)[:6] for name, p in prediction.items()})

The ERA5 decoder `"0"` also predicts `t_850` and `z_500`, on the O48 grid. As a **baseline** for the obs decoder, take the ERA5 forecast at the O48 point nearest to each sonde. The other baseline is the **climatology**: the mean of the observations, which is also the value the imputer fills in.

In [ ]:
from scipy.spatial import cKDTree


def xyz(lat, lon):
    lat, lon = np.deg2rad(lat), np.deg2rad(lon)
    return np.stack([np.cos(lat) * np.cos(lon), np.cos(lat) * np.sin(lon), np.sin(lat)], axis=-1)


nearest_o48 = cKDTree(xyz(datasets["data"].latitudes, datasets["data"].longitudes)).query(
    xyz(datasets["obs"].latitudes, datasets["obs"].longitudes))[1]

rows = {}
for v in OBSERVED:
    ok = ~np.isnan(truth[v])
    row = {"observations": ok.sum(),
           "obs decoder": np.sqrt(np.mean((prediction["obs"][v][ok] - truth[v][ok]) ** 2)),
           "climatology": np.sqrt(np.mean((datasets["obs"].statistics["mean"][datasets["obs"].name_to_index[v]] - truth[v][ok]) ** 2))}
    if v in prediction["data"]:
        row["ERA5 decoder (nearest point)"] = np.sqrt(np.mean((prediction["data"][v][nearest_o48][ok] - truth[v][ok]) ** 2))
    rows[v] = row
pd.DataFrame(rows).T

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
for ax, v in zip(axes, ["t_850", "z_500", "u_500"]):
    ok = ~np.isnan(truth[v])
    ax.scatter(truth[v][ok], prediction["obs"][v][ok], s=6, label="obs decoder")
    if v in prediction["data"]:
        ax.scatter(truth[v][ok], prediction["data"][v][nearest_o48][ok], s=6, alpha=0.5, label="ERA5 decoder")
    lims = [np.nanmin(truth[v][ok]), np.nanmax(truth[v][ok])]
    ax.plot(lims, lims, "k--", lw=1)
    ax.set(title=v, xlabel="observed", ylabel="predicted")
    ax.legend()
plt.tight_layout()

**Questions**
- How does the obs decoder compare with climatology? With the ERA5 decoder, after only 1000 steps?
- The pretrained model never saw `u` or `v`! Where can the obs decoder get the wind from? (Hint: geostrophic balance.)
- The radiosondes are not ERA5: they measure at a point, and have errors. Which RMSE would a perfect model get?

### 6.3 Decoding anywhere

The obs decoder predicts at every node of the obs grid, not only where there were soundings. Here is the predicted wind at 500 hPa everywhere, with the soundings on top: a quantity the pretrained model never had, now available globally.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 4.5), subplot_kw={"projection": ccrs.PlateCarree()})
lat, lon = datasets["obs"].latitudes, datasets["obs"].longitudes
for ax, v in zip(axes, ["u_500", "t_850"]):
    vmin, vmax = np.nanpercentile(prediction["obs"][v], [1, 99])
    im = scatter_map(ax, lat, lon, prediction["obs"][v], title=f"{v}: obs decoder (dots: soundings)", s=1,
                     cmap="RdBu_r" if v == "u_500" else "RdYlBu_r", vmin=vmin, vmax=vmax)
    scatter_map(ax, lat, lon, truth[v], title=f"{v}: obs decoder (dots: soundings)", s=14, cmap=im.cmap,
                vmin=vmin, vmax=vmax, edgecolors="k", linewidths=0.3)
    plt.colorbar(im, ax=ax, shrink=0.8)

On `main`, the places the decoder can predict at are the nodes of the obs dataset, fixed when the graph is built. Because the obs decoder has no trainable parameters per node or edge, the same weights would work with other nodes: decoding at **arbitrary locations** only needs a graph to them. With richer batches (section 8), the target locations are given with each sample, and the edges are built on the fly.

## 7. More ways to use observations

### 7.1 Encode observations (setup 2)

Now the observations are an **input**: a new encoder `obs` maps them to the hidden mesh, where its result is added to that of the ERA5 encoder (`model.latent_aggregator: SumAggregator`). The model gets the observations at *t*-6h and *t*: information that is not in its ERA5 input. All observed variables are forcings, so `obs` is an *input-only* dataset: no decoder, no loss.

In [ ]:
encode = load_config("encode_obs", OVERRIDES + FINETUNE)
pd.DataFrame(config_diff(pretrain, encode), columns=["setting", "pretrain_o48", "encode_obs"])

**Questions**
- In this setup the whole model is trained (no freezing), with a smaller learning rate. Why not freeze the pretrained model here?
- The imputer replaces a missing observation with the mean. How can the model tell a missing observation from one that is equal to the mean? What could we add to the dataset to help it?
- ERA5 is itself an analysis made with these radiosondes. What does this mean for what the model can gain from them here? What would be a more realistic setup for data assimilation?

To see if the observations help, compare with a model trained in the same way **without** observations: the pretrained model, trained for 1000 more steps. The two trainings take about 10 minutes together; you can also start them in a terminal (see `README.md`).

In [ ]:
run_training("encode_obs", OVERRIDES + FINETUNE)
run_training("pretrain_o48", OVERRIDES + FINETUNE + [
    "training/checkpoint/source=local", "training/checkpoint/loading=transfer_learning",
    "training.max_steps=1000", "training.optimization.lr=1e-4", "training.optimization.lr_scheduler.warmup_t=50",
    "diagnostics.log.mlflow.run_name=no-obs-control"])

In [ ]:
runs = [latest_run(client, "encode-obs"), latest_run(client, "no-obs-control")]
plot_metrics(client, runs, ["val_data_lossvariablemapper_loss", "val_data_mse_metric/data/z_500/1", "val_data_mse_metric/data/t_850/1"])

### 7.2 Observations only (setup 3)

An observation-only model, in the spirit of AIFS-DOP: no ERA5 at all. The same zarr is read as **two datasets**:
- `obs_in`: the observations at *t*-6h and *t*, only encoded (all variables are inputs, gaps filled by an `InputImputer`)
- `obs_out`: the observations at *t*+6h, only decoded (the observed variables are diagnostic, gaps masked by our `ObsImputer`)

Why two datasets? A single prognostic `obs` dataset would also work, but then the model's inputs and its targets would share the same processors, so filling in the inputs and masking the targets could not be set separately.

This model is trained **from scratch** (2000 steps, about 8 minutes), and the decoder output is a forecast on the whole O96 grid, learnt only from soundings.

In [ ]:
dop = load_config("dop_obs_only", OVERRIDES)
print("data.datasets:        ", list(dop.data.datasets))
print("graph.nodes:          ", list(dop.graph.nodes))
print("encoders:             ", {k: list(v.source_datasets) for k, v in dop.model.encoders.items()})
print("decoders:             ", {k: list(v.target_datasets) for k, v in dop.model.decoders.items()})
print("training_loss.datasets", list(dop.training.training_loss.datasets))

In [ ]:
run_training("dop_obs_only", OVERRIDES)

In [ ]:
dop_run = latest_run(client, "dop-obs-only")
dop_model = torch.load(OUTPUT_ROOT / "checkpoint" / dop_run.info.run_id / "inference-last.ckpt",
                       map_location="cuda", weights_only=False).cuda().eval()
dop_datasets = open_datasets({name: cfg.dataset_config for name, cfg in dop.dataloader.validation.datasets.items()})
dop_prediction = predict(dop_model, dop_datasets, forecast_date)

rows = {}
for v in OBSERVED:
    ok = ~np.isnan(truth[v])
    rows[v] = {"obs only (setup 3)": np.sqrt(np.mean((dop_prediction["obs_out"][v][ok] - truth[v][ok]) ** 2)),
               "ERA5 + obs decoder (setup 1)": np.sqrt(np.mean((prediction["obs"][v][ok] - truth[v][ok]) ** 2))}
display(pd.DataFrame(rows).T)

fig, ax = plt.subplots(figsize=(10, 4.5), subplot_kw={"projection": ccrs.PlateCarree()})
im = scatter_map(ax, lat, lon, dop_prediction["obs_out"]["z_500"], title="z_500 forecast of the observation-only model",
                 s=1, cmap="viridis")
plt.colorbar(im, ax=ax, shrink=0.8)

**Questions**
- Which variables does the observation-only model predict better than the model with ERA5 (setup 1), and which worse? Why could that be?
- Where is the forecast of the observation-only model good, and where not? Why?
- What other observations would it need? (The AIFS-DOP training data has many more: satellite radiances, surface stations, aircraft, ...)

## 8. Looking ahead: richer batches and dynamic observations

The next release of anemoi-core (branch `feat/richer-batch`, to be merged in the coming days) changes how data flows through anemoi, with observations in mind. What changes for you:

**Batches carry more than tensors.** On `main` a batch is a dictionary `{dataset name: tensor}`. With richer batches it is a `Batch` of `Source`s: each source has its `values` together with its `coordinates`, `variables`, `timedeltas` (the time of each value relative to the reference time) and statistics. Code that indexes the batch (custom callbacks, losses, tasks) uses `batch["obs"].values`.

**Tabular observations.** A dataset whose `dataset_config` has a `window` is read as a **tabular** dataset (`TabularDataReader`): each sample holds only the observations made in the time window, each with its own position and time. No binning onto a grid, no NaN gaps, no imputer, no mask:

```yaml
dataloader:
  training:
    datasets:
      surface_obs:
        dataset_config:
          dataset: ${system.input.surface_obs}
          frequency: ${data.frequency}
          window: (-6h,0]          # NEW: all observations of the 6 hours up to each time
          drop: [...]
```

**Graphs built on the fly.** The positions of tabular observations change from sample to sample, so the edges to and from them are built for every batch, with the same edge builders as before (`KNNEdges`, `CutOffEdges`, `ReversedKNNEdges`). The graph section is unchanged: whether edges are static or dynamic follows from the type of the dataset. These edges cannot have trainable parameters (`trainable_size: 0`, as we already used). New attributes `Timedeltas` give the nodes and edges the time of each observation.

**Decoding anywhere at inference.** `predict_step` takes a `target_template`: positions (and times) to decode at, without data. Setup 1 is then a model that predicts the observed quantities at any location.

**Config changes** (among others):

| `main` | richer batches |
|---|---|
| `dataset_fusing_strategy: "not_supported"` | `"none"` or `"sequential"` (several datasets share one encoder) + `node_embedder` |
| decoder target features: `coordinates`, `forcings`, ... | also `target_forcings`: forcings at the *output* time and location (e.g. the satellite viewing angles of an observation) |
| `NaNMaskScaler` (`nan_mask_weights`), imputer for gaps in targets | removed: use `ignore_nans: True` on the loss |
| any DDP strategy | with tabular datasets: `static_graph: false`, `find_unused_parameters: true` |

A full example with ERA5 and seven observation types (encoded and decoded) is in `training/tests/config/test_single.yaml` of anemoi-core (branch `feat/richer-batch`).

**From this notebook to dynamic observations**

| | `main` (this notebook) | richer batches |
|---|---|---|
| observation dataset | binned onto a grid, NaN where there is no observation | the observations themselves, with `window` |
| dates | same dates as ERA5 | observations from a time window around each ERA5 date |
| obs nodes | fixed: the grid points | the observations of each sample |
| gaps | `ObsImputer` + `nan_mask_weights` | none |
| setup 1: decode | `decoders.obs`, `target_node_features: [coordinates, forcings]` | the same, possibly with `target_forcings` |
| setup 2: encode | `encoders.obs`, all observed variables as forcings | the same, `trainable_size: 0` |
| setup 3: obs only | `obs_in` + `obs_out` | the same |

## 9. Tasks

### Task 1: fine-tune more
Fine-tune the whole model instead of the obs decoder only, with a smaller learning rate:
```
run_training("finetune_obs_decoder", OVERRIDES + FINETUNE + ["training.checkpoint.modifiers.0.submodules_to_freeze=[]",
             "training.optimization.lr=1e-4", "diagnostics.log.mlflow.run_name=finetune-all"])
```
Does the obs decoder get better? What happens to the ERA5 validation loss?

### Task 2: what does the decoder need to know?
Change `model.decoders.obs.target_node_features`, e.g. to `["coordinates"]` only, or add `"trainable_parameters"` (and set `model.node_trainable_parameters.obs: 8`). How do the results change? Which of these models could decode at new locations?

### Task 3: other observations
Add more observed variables (other levels, `q`), or use another observation dataset: copy `configs/obs/radiosondes.yaml`, change the path and the variables, and select it with the override `obs=<your file name>`. For example, satellite radiances binned on the same grid (`aifsdop-ea-ofb-oper-0001-mars-o96-2012-2024-6h-v1-observations-atms-radiances`). Mind the dates!

### Task 4: encode and decode
Combine setups 1 and 2: observations at *t*-6h and *t* as input, and observations at *t*+6h as target (two datasets, as in setup 3, plus ERA5).